## Load data for the stratified analysis

This notebook holds the analysis that sits on top of the stratified rebuild: RQ1 and the comparison table, both using the stratified folds with code 15 held out. RQ2 is in Notebook 10 and the attribution work is in Notebook 11. The MT-TrajNet numbers used here are read from the authoritative run in Notebook 8 and are not recomputed.

Two folders are used and they are kept separate on purpose. DATA is the raw dataset folder and is only ever read from. RESULTS is the results folder inside the repository and is the only place this notebook writes to, so the saved files and this notebook can never drift apart. Both are checked before anything runs.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA=Path(r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets")
REPO=Path(r"C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis")
RESULTS=REPO/"results"

assert DATA.exists(),f"data folder not found: {DATA}"
assert RESULTS.exists(),f"results folder not found: {RESULTS}"

proc=pd.read_csv(DATA/"Process.csv",sep=";")
lab=pd.read_csv(DATA/"Laboratory.csv",sep=";")
df=proc.merge(lab,on="batch",suffixes=("","_lab"))

targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
print("reading data from:",DATA)
print("writing results to:",RESULTS)
print()
print("proc shape:",proc.shape,"| lab shape:",lab.shape,"| merged:",df.shape)
print("targets present:",[t for t in targets if t in df.columns])
print("has code column:","code" in df.columns)
print("row order preserved by merge:",bool((df["batch"].values==proc["batch"].values).all()))

reading data from: C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets
writing results to: C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results

proc shape: (1005, 35) | lab shape: (1005, 55) | merged: (1005, 89)
targets present: ['dissolution_av', 'tbl_av_hardness', 'tbl_rsd_weight', 'fct_tensile']
has code column: True
row order preserved by merge: True


## Stratified folds

I rebuild the same stratified fold assignment used in the authoritative run: code 15 held out for the out-of-distribution test, the four remaining high-hardness codes distributed so the high-cluster counts per fold are 22, 6 and 6. Code 4 carries 22 batches on its own and cannot be split without breaking the grouping, so 22/6/6 is the best balance achievable while keeping codes intact. It is a large improvement on the plain GroupKFold split, which placed 90 of the 98 high-cluster batches in a single fold. I confirm the distribution before using these folds for any comparison.

In [2]:
from collections import Counter

codes=df["code"].values
OOD_CODE=15
fold_codes=[
[1,2,4,6,7,9,13,18,25],
[10,11,17,19,22,24],
[3,5,8,12,14,16,20,21,23],
]
high_in_cv=[4,8,20,24]
counts=Counter(int(c) for c in codes)

print("OOD held-out code:",OOD_CODE,f"({counts[OOD_CODE]} batches)")
print()
allcv=[]
for f in range(3):
    fc=fold_codes[f]
    hi=[c for c in fc if c in high_in_cv]
    hi_b=sum(counts[c] for c in hi)
    tot=sum(counts[c] for c in fc)
    print(f"fold {f+1}: {tot} batches | high-cluster {hi} -> {hi_b} batches")
    allcv+=fc

print()
print("check: high-cluster per fold is 22/6/6:",
      [sum(counts[c] for c in fold_codes[f] if c in high_in_cv) for f in range(3)]==[22,6,6])
print("check: code 15 excluded:",OOD_CODE not in allcv)
print("check: no code repeated:",len(allcv)==len(set(allcv)))
print("check: CV + OOD = 25:",len(set(allcv))+1==25)
print("check: CV batches total 941:",sum(counts[c] for c in allcv)==941)

OOD held-out code: 15 (64 batches)

fold 1: 314 batches | high-cluster [4] -> 22 batches
fold 2: 313 batches | high-cluster [24] -> 6 batches
fold 3: 314 batches | high-cluster [8, 20] -> 6 batches

check: high-cluster per fold is 22/6/6: True
check: code 15 excluded: True
check: no code repeated: True
check: CV + OOD = 25: True
check: CV batches total 941: True


## RQ1: MT-TrajNet against tuned XGBoost

I compare MT-TrajNet against the tuned XGBoost configuration (500 trees, depth 5, learning rate 0.03, subsample 0.8) on each target, using the same stratified folds with code 15 excluded from both. This is the same configuration reported as xgb_tuned in the comparison table below, so the two tables agree.

XGBoost is trained and evaluated on identical folds so the comparison is paired. The MT-TrajNet fold values and means are taken from the authoritative run in Notebook 8 and are not recomputed here. The Nadeau-Bengio corrected test adjusts for the overlap between folds; with three folds it has low power, so p-values are weak evidence in either direction.

In [3]:
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from scipy import stats
import json

leak=["batch","code","code_lab","strength","size","start","Drug release average (%)","Drug release min (%)",
      "Residual solvent","Total impurities","Impurity O","Impurity L"]+targets
Xf=proc.drop(columns=[c for c in leak if c in proc.columns]).copy()
if "weekend" in Xf.columns:
    Xf["weekend"]=Xf["weekend"].map({"no":0,"yes":1}).astype(int)

mttraj_folds={
"dissolution_av":[3.344,3.147,3.292],
"tbl_av_hardness":[7.900,7.428,7.767],
"tbl_rsd_weight":[0.770,0.442,0.543],
"fct_tensile":[0.404,0.350,0.283],
}
mttraj_mean_auth={"dissolution_av":3.261,"tbl_av_hardness":7.698,"tbl_rsd_weight":0.585,"fct_tensile":0.345}

xgb_folds={t:[] for t in targets}
for t in targets:
    yt=df[t].values
    for f in range(3):
        te=np.isin(codes,fold_codes[f])
        tr=np.isin(codes,[c for ff in range(3) if ff!=f for c in fold_codes[ff]])
        m=XGBRegressor(n_estimators=500,max_depth=5,learning_rate=0.03,subsample=0.8,random_state=42,verbosity=0)
        m.fit(Xf[tr],yt[tr])
        xgb_folds[t].append(np.sqrt(mean_squared_error(yt[te],m.predict(Xf[te]))))

print(f"{'target':<18}{'XGB_tuned':<11}{'MT-TrajNet':<12}{'diff':<9}{'p-value':<10}{'significant'}")
rq1={}
for t in targets:
    x=np.array(xgb_folds[t]);mt=np.array(mttraj_folds[t])
    d=mt-x
    n=len(d);mean_d=d.mean();var_d=d.var(ddof=1)
    cv=var_d*(1/n+0.5)
    tstat=mean_d/np.sqrt(cv) if cv>0 else 0
    p=2*(1-stats.t.cdf(abs(tstat),n-1))
    xgb_mean=round(float(x.mean()),3)
    disp_diff=round(mttraj_mean_auth[t]-xgb_mean,3)
    rq1[t]={"xgb_folds":[round(float(v),3) for v in x],"xgb_mean":xgb_mean,
            "mt_folds":list(mt),"mt_mean":mttraj_mean_auth[t],"diff":disp_diff,
            "p":round(float(p),3),"significant":bool(p<0.05)}
    print(f"{t:<18}{xgb_mean:<11.3f}{mttraj_mean_auth[t]:<12.3f}{disp_diff:<9.3f}{p:<10.3f}{'yes' if p<0.05 else 'no'}")

with open(RESULTS/"rq1_stratified.json","w") as fh:
    json.dump({"comparison":"MT-TrajNet (stratified folds, cross-fitting) vs tuned XGBoost (500 trees, depth 5, lr 0.03, subsample 0.8), same stratified folds, code 15 excluded, Nadeau-Bengio corrected paired test",
               "note":"three folds gives low power; p-values are weak evidence in either direction. Fold values in this file are rounded to three decimals for storage: the MT-TrajNet folds as saved by Notebook 8, the XGBoost folds on writing. All means and p-values are computed from the unrounded values, so recomputing them from the stored folds can differ by up to 0.002. mt_mean is Notebook 8's mean from the unrounded folds.",
               "results":rq1},fh,indent=2)
print("\nsaved",RESULTS/"rq1_stratified.json")

target            XGB_tuned  MT-TrajNet  diff     p-value   significant
dissolution_av    3.243      3.261       0.018    0.886     no
tbl_av_hardness   4.761      7.698       2.937    0.015     yes
tbl_rsd_weight    0.611      0.585       -0.026   0.697     no
fct_tensile       0.270      0.345       0.075    0.269     no

saved C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results\rq1_stratified.json


## Comparison table on the stratified folds

For a fair comparison every model must be evaluated on the same folds. I recompute the classical baselines on the same stratified folds with code 15 excluded, then place the stratified MT-TrajNet results alongside them, so every number in the table comes from the same evaluation setup.

Two configurations are reported for LASSO and for XGBoost: the default settings used in the earlier baseline work, and a second configuration with more capacity (XGBoost 500 trees, depth 5, learning rate 0.03, subsample 0.8; LASSO alpha 0.01). These are labelled tuned in the table for consistency with the earlier work, but they were selected by hand rather than by a grid search, so the label should be read as a second configuration rather than a full hyperparameter search.

The MT-TrajNet column is the authoritative mean from Notebook 8 and is not re-run here. The best_baseline column is the lowest RMSE across the classical models, excluding the dummy.

In [4]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def cv_rmse(model_fn,yt):
    rmses=[]
    for f in range(3):
        te=np.isin(codes,fold_codes[f])
        tr=np.isin(codes,[c for ff in range(3) if ff!=f for c in fold_codes[ff]])
        m=model_fn()
        m.fit(Xf[tr],yt[tr])
        rmses.append(np.sqrt(mean_squared_error(yt[te],m.predict(Xf[te]))))
    return np.mean(rmses)

models={
"dummy":lambda:DummyRegressor(strategy="mean"),
"lasso":lambda:make_pipeline(StandardScaler(),Lasso(alpha=0.1,max_iter=5000)),
"xgb":lambda:XGBRegressor(n_estimators=300,max_depth=4,learning_rate=0.05,random_state=42,verbosity=0),
"lasso_tuned":lambda:make_pipeline(StandardScaler(),Lasso(alpha=0.01,max_iter=5000)),
"xgb_tuned":lambda:XGBRegressor(n_estimators=500,max_depth=5,learning_rate=0.03,subsample=0.8,random_state=42,verbosity=0),
}

mttraj_mean={"dissolution_av":3.261,"tbl_av_hardness":7.698,"tbl_rsd_weight":0.585,"fct_tensile":0.345}

rows={}
for t in targets:
    yt=df[t].values
    row={name:round(cv_rmse(fn,yt),3) for name,fn in models.items()}
    row["MT_TrajNet"]=mttraj_mean[t]
    rows[t]=row

comparison=pd.DataFrame(rows).T
basecols=[c for c in comparison.columns if c not in ["dummy","MT_TrajNet"]]
comparison["best_baseline"]=comparison[basecols].min(axis=1)
comparison=comparison.round(3)

print("COMPARISON TABLE (stratified folds, code 15 excluded, RMSE, lower is better)")
print(comparison)

comparison.to_csv(RESULTS/"comparison_stratified.csv")
print("\nsaved",RESULTS/"comparison_stratified.csv")
print()
print("cross-check: RQ1 XGB_tuned column must equal the xgb_tuned column above")
for t in targets:
    a=rq1[t]["xgb_mean"];b=comparison.loc[t,"xgb_tuned"]
    print(f"  {t:<18}RQ1 {a:<8.3f}table {b:<8.3f}{'match' if abs(a-b)<1e-9 else 'MISMATCH'}")

COMPARISON TABLE (stratified folds, code 15 excluded, RMSE, lower is better)
                 dummy  lasso    xgb  lasso_tuned  xgb_tuned  MT_TrajNet  \
dissolution_av   3.533  3.171  3.272        3.205      3.243       3.261   
tbl_av_hardness  8.820  5.962  4.737        6.087      4.761       7.698   
tbl_rsd_weight   0.573  0.557  0.635        0.563      0.611       0.585   
fct_tensile      0.422  0.359  0.279        0.325      0.270       0.345   

                 best_baseline  
dissolution_av           3.171  
tbl_av_hardness          4.737  
tbl_rsd_weight           0.557  
fct_tensile              0.270  

saved C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results\comparison_stratified.csv

cross-check: RQ1 XGB_tuned column must equal the xgb_tuned column above
  dissolution_av    RQ1 3.243   table 3.243   match
  tbl_av_hardness   RQ1 4.761   table 4.761   match
  tbl_rsd_weight    RQ1 0.611   table 0.611   match
  fct_tensile       RQ1 0.270   table 0.270   match
